# Mutation Analysis Demo

Demonstration of industry-standard mutation analysis with statistics and experimental validation.

This notebook shows how to:
1. Run mutations with statistical analysis (multiple runs, error bars, convergence metrics)
2. Compare predictions to experimental ΔΔG values from literature
3. Visualize results with publication-ready plots
4. Perform position scanning with heatmap visualization
5. Conduct alanine scanning to identify hotspot residues

**Test System:** Barnase (1BNI)
- Small, stable bacterial ribonuclease
- Extensively studied for protein stability
- Rich experimental mutagenesis data

**Experimental Data Source:**
Serrano, L., Kellis, J.T., Cann, P., Matouschek, A., & Fersht, A.R. (1993).
"Step-wise Mutation of Barnase to Binase"
*J. Mol. Biol.* 233, 305-312.

**Requirements:**
- OpenMM and PDBFixer: `pip install sicifus[energy]`

**Download test structure:**
```bash
curl -o 1BNI.pdb https://files.rcsb.org/download/1BNI.pdb
```

## Setup

Import required modules and download the Barnase structure.

In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sicifus import MutationEngine
from sicifus.visualization import plot_ddg, plot_position_scan_heatmap, plot_alanine_scan

# Initialize mutation engine
engine = MutationEngine()

In [2]:
# Download and clean Barnase structure
import urllib.request
from pathlib import Path

if not Path("1BNI.pdb").exists():
    print("Downloading Barnase structure (1BNI) from RCSB...")
    urllib.request.urlretrieve("https://files.rcsb.org/download/1BNI.pdb", "1BNI_raw.pdb")
    
    print("Cleaning structure (removing waters and heteroatoms)...")
    with open("1BNI_raw.pdb", "r") as f_in, open("1BNI.pdb", "w") as f_out:
        for line in f_in:
            # Keep only ATOM records (skip HETATM for waters, ligands)
            if line.startswith("ATOM"):
                f_out.write(line)
            # Keep structural records
            elif line.startswith(("MODEL", "ENDMDL", "END", "TER")):
                f_out.write(line)
    
    print("✅ Downloaded and cleaned 1BNI.pdb")
else:
    print("Using existing 1BNI.pdb")

Using existing 1BNI.pdb


## Demo 1: Single Mutation with Statistics

Run a single mutation (H18K - Histidine to Lysine at position 18) with statistical analysis.

**Experimental value:** H18K has ΔΔG = +1.19 kcal/mol (Serrano et al. 1993, Table 3)
- Destabilizing mutation (His to Lys in α-helix 1)
- Located in first α-helix

By running multiple independent minimizations (`n_runs=5`), we can estimate:
- **Standard deviation** of the energy prediction
- **95% confidence interval** for ΔΔG
- **Convergence metric** (coefficient of variation)

In [3]:
# Run mutation with 5 independent minimizations
result = engine.mutate(
    "1BNI.pdb",
    ["H18K"],
    n_runs=1,              # Multiple runs for statistics
    constrain_backbone=True,
    keep_statistics=True,  # Store all run data
    use_mean=False,        # Report best energy (default)
    max_iterations=10    # Minimization steps
)

In [4]:
# Display results
label = "H18K"
print(f"Mutation: {label}")
print(f"Wild-type energy: {result.wt_energy:.2f} kcal/mol")
print(f"Mutant energy (best): {result.mutant_energies[label]:.2f} kcal/mol")
print(f"ΔΔG (best-based): {result.ddg[label]:+.2f} kcal/mol")
print()

# Statistical summary
if result.mean_energy:
    print(f"Statistical Summary (n={len(result.all_run_energies[label])} runs):")
    print(f"  Mean energy: {result.mean_energy[label]:.2f} ± {result.sd_energy[label]:.2f} kcal/mol")
    print(f"  Range: [{result.min_energy[label]:.2f}, {result.max_energy[label]:.2f}]")
    print(f"  ΔΔG (mean-based): {result.ddg_mean[label]:+.2f} ± {result.ddg_sd[label]:.2f} kcal/mol")
    print(f"  95% CI: [{result.ddg_ci_95[label][0]:+.2f}, {result.ddg_ci_95[label][1]:+.2f}]")
    print(f"  Convergence (CV): {result.convergence_metric[label]:.3f}")
    print()

# Compare to experiment
experimental_ddg = 1.19
predicted_ddg = result.ddg[label]
error = predicted_ddg - experimental_ddg

print("Comparison to Experiment:")
print(f"  Experimental ΔΔG: {experimental_ddg:+.2f} kcal/mol (Serrano et al. 1993)")
print(f"  Predicted ΔΔG:    {predicted_ddg:+.2f} kcal/mol")
print(f"  Error:            {error:+.2f} kcal/mol")
print(f"  Absolute error:   {abs(error):.2f} kcal/mol")

Mutation: H18K
Wild-type energy: 6413.67 kcal/mol
Mutant energy (best): 6378.11 kcal/mol
ΔΔG (best-based): -35.55 kcal/mol

Comparison to Experiment:
  Experimental ΔΔG: +1.19 kcal/mol (Serrano et al. 1993)
  Predicted ΔΔG:    -35.55 kcal/mol
  Error:            -36.74 kcal/mol
  Absolute error:   36.74 kcal/mol


**Interpretation:**
- **ΔΔG > 0**: Mutation destabilizes the protein (unfavorable)
- **ΔΔG < 0**: Mutation stabilizes the protein (favorable)
- **CV < 0.1**: Good convergence (results are reproducible)
- **Error < 1 kcal/mol**: Acceptable prediction accuracy

## Demo 2: Batch Mutations with Experimental Validation

Run a set of well-characterized Barnase mutations and compare to experimental values.

**Dataset:** Mutations from Serrano et al. (1993) - *J Mol Biol* 233:305-312, Table 3
- These are single-point mutations at various positions
- Experimental ΔΔG measured by urea denaturation
- Temperature: 25°C, pH: 6.3

In [5]:
# Define mutations with experimental data from Serrano et al. (1993), Table 3
mutations_df = pl.DataFrame({
    "mutation": ["H18K", "I55V", "K62R", "K66A", "T79V", "S85A", "I88L", "L89V"],
    "chain": ["A"] * 8,
    "experimental_ddg": [1.19, 0.29, 0.48, -0.25, -0.29, 0.12, 0.28, 0.27],
    "location": ["α-helix 1", "β-strand 1", "Loop", "α-helix 2", "Loop", "Loop", "β-strand 3", "β-strand 3"],
    "notes": [
        "His to Lys - destabilizing",
        "Ile to Val - smaller hydrophobic",
        "Lys to Arg - guanosine binding loop",
        "Lys to Ala - stabilizing",
        "Thr to Val - stabilizing",
        "Ser to Ala - slight destabilizing",
        "Ile to Leu - conservative",
        "Leu to Val - conservative"
    ]
})

print("Mutations to analyze:")
mutations_df.select(["mutation", "experimental_ddg", "location", "notes"])

Mutations to analyze:


mutation,experimental_ddg,location,notes
str,f64,str,str
"""H18K""",1.19,"""α-helix 1""","""His to Lys - destabilizing"""
"""I55V""",0.29,"""β-strand 1""","""Ile to Val - smaller hydrophob…"
"""K62R""",0.48,"""Loop""","""Lys to Arg - guanosine binding…"
"""K66A""",-0.25,"""α-helix 2""","""Lys to Ala - stabilizing"""
"""T79V""",-0.29,"""Loop""","""Thr to Val - stabilizing"""
"""S85A""",0.12,"""Loop""","""Ser to Ala - slight destabiliz…"
"""I88L""",0.28,"""β-strand 3""","""Ile to Leu - conservative"""
"""L89V""",0.27,"""β-strand 3""","""Leu to Val - conservative"""


In [ ]:
# Run batch mutations
print("Running mutations (this will take several minutes)...\n")

results_df = engine.mutate_batch(
    "1BNI.pdb",
    mutations_df,
    n_runs=3,
    max_iterations=2000,
    keep_statistics=True,
)

# Add predicted_ddg column (alias for ddg_kcal_mol for clarity)
results_df = results_df.with_columns(
    pl.col("ddg_kcal_mol").alias("predicted_ddg")
)

print("\nResults:")
results_df.select(["mutation", "experimental_ddg", "predicted_ddg", "location"])

Running mutations (this will take several minutes)...

Preparing wild-type structure (repair-once)...


### Calculate Prediction Metrics

In [ ]:
# Extract arrays (excluding any NaN predictions)
valid_results = results_df.filter(~pl.col("predicted_ddg").is_nan())

experimental = valid_results["experimental_ddg"].to_numpy()
predicted = valid_results["predicted_ddg"].to_numpy()

# Calculate metrics
r_squared = stats.pearsonr(experimental, predicted)[0] ** 2
rmse = np.sqrt(np.mean((predicted - experimental) ** 2))
mae = np.mean(np.abs(predicted - experimental))
pearson_r, pearson_p = stats.pearsonr(experimental, predicted)

print("="*60)
print("PREDICTION PERFORMANCE")
print("="*60)
print(f"Sample size:       {len(experimental)} mutations")
print(f"R² (R-squared):    {r_squared:.3f}")
print(f"RMSE:              {rmse:.2f} kcal/mol")
print(f"MAE:               {mae:.2f} kcal/mol")
print(f"Pearson r:         {pearson_r:.3f} (p = {pearson_p:.2e})")
print("="*60)

# Interpretation
print("\nINTERPRETATION:")
if r_squared > 0.5:
    print(f"✅ Good correlation (R² = {r_squared:.3f})")
elif r_squared > 0.3:
    print(f"⚠️  Moderate correlation (R² = {r_squared:.3f})")
else:
    print(f"❌ Weak correlation (R² = {r_squared:.3f})")

if mae < 1.0:
    print(f"✅ Low MAE ({mae:.2f} kcal/mol - good accuracy)")
elif mae < 1.5:
    print(f"⚠️  Moderate MAE ({mae:.2f} kcal/mol - acceptable)")
else:
    print(f"❌ High MAE ({mae:.2f} kcal/mol - poor accuracy)")

print("\nContext: State-of-the-art methods typically achieve:")
print("  - R² ~ 0.4-0.7 for diverse mutations")
print("  - MAE ~ 0.8-1.2 kcal/mol")

### Detailed Comparison Table

In [ ]:
# Add error column
comparison_df = valid_results.with_columns(
    (pl.col("predicted_ddg") - pl.col("experimental_ddg")).alias("error"),
    (pl.col("predicted_ddg") - pl.col("experimental_ddg")).abs().alias("abs_error")
)

print("Detailed Comparison:")
print()
comparison_df.select([
    "mutation", 
    "experimental_ddg", 
    "predicted_ddg", 
    "error", 
    "abs_error",
    "location"
]).sort("abs_error", descending=True)

### Visualization: Predicted vs Experimental Scatter Plot

In [ ]:
# Create scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

# Scatter plot
ax.scatter(experimental, predicted, alpha=0.7, s=150, 
           edgecolors='black', linewidth=1.5, c='steelblue')

# Perfect prediction line (y = x)
min_val = min(experimental.min(), predicted.min()) - 0.5
max_val = max(experimental.max(), predicted.max()) + 0.5
ax.plot([min_val, max_val], [min_val, max_val], 
        'k--', alpha=0.5, linewidth=2, label='Perfect prediction')

# Linear regression fit
slope, intercept = np.polyfit(experimental, predicted, 1)
fit_x = np.linspace(min_val, max_val, 100)
fit_y = slope * fit_x + intercept
ax.plot(fit_x, fit_y, 'r-', alpha=0.7, linewidth=2,
        label=f'Linear fit (y = {slope:.2f}x + {intercept:+.2f})')

# Labels
ax.set_xlabel('Experimental ΔΔG (kcal/mol)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted ΔΔG (kcal/mol)', fontsize=12, fontweight='bold')
ax.set_title('Barnase Mutations: Predicted vs Experimental',
             fontsize=14, fontweight='bold', pad=15)

# Statistics text box
stats_text = (
    f"n = {len(experimental)}\n"
    f"R² = {r_squared:.3f}\n"
    f"RMSE = {rmse:.2f} kcal/mol\n"
    f"MAE = {mae:.2f} kcal/mol\n"
    f"Pearson r = {pearson_r:.3f}"
)

ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
        fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, pad=0.8))

# Formatting
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='lower right', fontsize=10)
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig("barnase_validation.png", dpi=300, bbox_inches='tight')
print("Saved to barnase_validation.png")
plt.show()

### Visualize ΔΔG with Error Bars

In [ ]:
# Create bar chart with experimental comparison
fig, ax = plt.subplots(figsize=(12, 6))

mutations = comparison_df["mutation"].to_list()
exp_values = comparison_df["experimental_ddg"].to_numpy()
pred_values = comparison_df["predicted_ddg"].to_numpy()

x = np.arange(len(mutations))
width = 0.35

# Plot bars
bars1 = ax.bar(x - width/2, exp_values, width, label='Experimental',
               color='#2ca02c', edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, pred_values, width, label='Predicted (Sicifus)',
               color='#1f77b4', edgecolor='black', linewidth=1)

# Add error bars if statistics available
if "ddg_sd" in comparison_df.columns:
    pred_sd = comparison_df["ddg_sd"].to_numpy()
    ax.errorbar(x + width/2, pred_values, yerr=pred_sd, 
                fmt='none', ecolor='black', capsize=3, linewidth=1)

# Formatting
ax.set_xlabel('Mutation', fontsize=12, fontweight='bold')
ax.set_ylabel('ΔΔG (kcal/mol)', fontsize=12, fontweight='bold')
ax.set_title('Barnase Mutations: Experimental vs Predicted ΔΔG', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(mutations, rotation=45, ha='right')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("barnase_comparison.png", dpi=300, bbox_inches='tight')
print("Saved to barnase_comparison.png")
plt.show()

## Demo 3: Position Scan with Heatmap

Scan all 20 amino acids at selected positions to create a comprehensive mutational landscape.

**Note:** This is computationally expensive (19 mutations per position). We'll scan just 3 positions as an example.

In [ ]:
# Scan positions 24, 35, 56 (all 20 amino acids each)
# These positions have known experimental data for some mutations
scan_df = engine.position_scan(
    "1BNI.pdb",
    chain="A",
    positions=[24, 35, 56],
    max_iterations=1500,
    constrain_backbone=True,  # Keep backbone fixed for speed
)

print(f"Completed {len(scan_df)} mutations")
print("\nSample results:")
scan_df.head(10)

In [ ]:
# Create heatmap visualization
plot_position_scan_heatmap(
    scan_df,
    output_file="barnase_position_scan.png",
    cmap="RdBu_r",  # Red (destabilizing) to Blue (stabilizing)
    vmin=-2.0,
    vmax=3.0
)

print("Saved to barnase_position_scan.png")

**Heatmap interpretation:**
- **Rows**: Mutant amino acid type (all 20 possibilities)
- **Columns**: Position in the protein
- **Color**: Red = destabilizing, Blue = stabilizing, White = neutral
- **Black boxes**: Wild-type residue at each position

**Expected patterns:**
- Position 35 (Trp): Most mutations destabilizing (large aromatic in core)
- Position 56 (Phe): Similar pattern, slightly more permissive
- Position 24 (Ala→Gly): Should show G as slightly destabilizing

## Demo 4: Alanine Scan

Alanine scanning is a classic method to identify "hotspot" residues - positions where mutation to alanine significantly impacts stability.

In [ ]:
# Perform alanine scan on positions 20-30 (includes several from our validation set)
scan_df = engine.alanine_scan(
    "1BNI.pdb",
    chain="A",
    positions=list(range(20, 31)),
    max_iterations=1500,
)

print("Alanine scan results (sorted by ddG):")
scan_df.sort("ddg_kcal_mol", descending=True)

In [ ]:
# Visualize with sorted bar chart
plot_alanine_scan(
    scan_df,
    output_file="barnase_alanine_scan.png",
    highlight_threshold=1.5  # Highlight if |ΔΔG| > 1.5 kcal/mol
)

print("Saved to barnase_alanine_scan.png")

**Hotspot identification:**
- Positions with large positive ΔΔG are critical for stability
- Expected hotspots: Hydrophobic core residues (Ile, Leu, Val in buried regions)
- Compare to experimental: H18K = +1.19 kcal/mol (destabilizing when changed)

## Summary

This notebook demonstrated:

1. ✅ **Statistical rigor** - Multiple runs with error estimates
2. ✅ **Experimental validation** - Direct comparison to literature data
3. ✅ **Performance metrics** - R², RMSE, MAE to assess accuracy
4. ✅ **Comprehensive scanning** - Position scans and alanine scans
5. ✅ **Publication plots** - Heatmaps, bar charts, scatter plots

### Key Takeaways

- **Barnase is a good test system** - Well-characterized, stable structure
- **Accuracy is method-dependent** - Force fields have inherent limitations (~1 kcal/mol error)
- **Statistics matter** - Multiple runs reveal prediction uncertainty
- **Experimental validation is critical** - Always compare to known data when possible

### Next Steps

- Explore interface mutations: `interface_analysis_demo.ipynb`
- Larger validation dataset: `experimental_validation_demo.ipynb`
- Check the [documentation](https://arikat.github.io/sicifus) for advanced features

## Your Experiments

Use the cells below to experiment with your own structures and mutations:

In [ ]:
# Your code here
